# Extract findings — pull one small results folder out of a big published output

**Problem this solves.** A GPU notebook's output can be tens of GB, almost all of
it checkpoints, while the part worth reading is a few hundred KB of CSV and JSON.
Kaggle's output browser downloads *one file* or *the entire tree* — nothing in
between — so a folder only becomes downloadable by becoming a file.

**What this does.** Mount the published output as a dataset, find the findings
folder inside it, copy only the small files into this notebook's own output,
zip them into one file, and print the handoff digest to the console.

**Accelerator: None.** This is file copying; a GPU here would burn quota for
nothing. Set the accelerator to *None* before running.

**Fastest path of all:** run it, then copy the `CLAUDE-HANDOFF-JSON` block the
last cell prints. Nothing needs downloading at all.

In [ ]:
# ============================================================
# CELL 1 - FIND, FILTER, COPY, ZIP
# ============================================================
import os, sys, json, shutil, hashlib
from pathlib import Path

# Which findings folder to extract. None = take whatever is in the dataset (and
# list them all if there is more than one).
WANT = "04-whisper-core-sanity"

# Anything bigger than this, or carrying a weights extension, is refused. The
# whole point is to not move model weights around.
MAX_FILE_MB = 50
TEXT_SUFFIXES = {".json", ".csv", ".txt", ".md", ".tsv", ".log", ".yaml", ".yml"}
WEIGHT_SUFFIXES = {".safetensors", ".bin", ".pt", ".pth", ".ckpt", ".h5", ".onnx"}

INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")

def head(t):
    print("\n" + "=" * 74); print(t); print("=" * 74)

head("DISCOVERY")
# Content-based, as everywhere else in this project: a findings folder is the
# one holding dataset_findings.json. Never a hardcoded dataset slug or depth -
# Kaggle mounts owner-scoped and the archive root often carries an extra level.
hits = sorted(INPUT.rglob("dataset_findings.json"))
if not hits:
    hits = sorted(p / "dataset_findings.json" for p in INPUT.rglob("findings/*") if p.is_dir())
    hits = [p for p in hits if p.exists()]
cands = [p.parent for p in hits]

print("findings folders visible in /kaggle/input:")
for c in cands:
    n = sum(1 for x in c.rglob("*") if x.is_file())
    b = sum(x.stat().st_size for x in c.rglob("*") if x.is_file())
    print("  %-28s %3d files  %8.1f KB   %s" % (c.name, n, b / 1024, c))
if not cands:
    raise RuntimeError(
        "No findings folder found under /kaggle/input. Attach the dataset you "
        "published from the training notebook's output, then re-run.")

SRC = None
if WANT:
    SRC = next((c for c in cands if c.name == WANT), None)
    if SRC is None:
        print("\n!! WANT = %r not found; falling back to the only/first candidate." % WANT)
if SRC is None:
    SRC = cands[0]
RUN_ID = SRC.name
print("\nextracting: %s\nfrom      : %s" % (RUN_ID, SRC))

head("COPY (small text artifacts only)")
DEST = WORKING / "findings" / RUN_ID
shutil.rmtree(DEST, ignore_errors=True)
DEST.mkdir(parents=True, exist_ok=True)

copied, skipped, total = [], [], 0
for p in sorted(SRC.rglob("*")):
    if not p.is_file():
        continue
    rel = p.relative_to(SRC)
    mb = p.stat().st_size / 1e6
    if p.suffix.lower() in WEIGHT_SUFFIXES:
        skipped.append((rel, mb, "model weights")); continue
    if mb > MAX_FILE_MB:
        skipped.append((rel, mb, "over %d MB" % MAX_FILE_MB)); continue
    if p.suffix.lower() not in TEXT_SUFFIXES:
        skipped.append((rel, mb, "not a text artifact")); continue
    (DEST / rel).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, DEST / rel)
    copied.append((rel, mb))
    total += p.stat().st_size

for rel, mb in copied:
    print("  copied  %-42s %8.1f KB" % (rel, mb * 1000))
for rel, mb, why in skipped:
    print("  SKIPPED %-42s %8.1f MB   (%s)" % (rel, mb, why))
print("\n%d files copied, %.1f KB total" % (len(copied), total / 1024))

ZIP = shutil.make_archive(str(WORKING / (RUN_ID + "-lite")), "zip",
                          root_dir=str(DEST.parent), base_dir=RUN_ID)
print("one-file download -> %s  (%.1f KB)" % (ZIP, Path(ZIP).stat().st_size / 1024))
print("")
print("Output tab -> download %s alone. Or skip downloading entirely and copy" % Path(ZIP).name)
print("the digest block the next cell prints.")

## The digest, and the numbers worth reading first

`dataset_findings.json` is the same content as the `CLAUDE-HANDOFF-JSON` block
the original run printed. Copying it back into the chat is the whole handoff —
no download required.

In [ ]:
# ============================================================
# CELL 2 - SUMMARY + DIGEST
# ============================================================
D = {}
dj = DEST / "dataset_findings.json"
if dj.exists():
    D = json.loads(dj.read_text(encoding="utf-8"))

def get(path, default=None):
    cur = D
    for k in path.split("."):
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

head("WHAT THIS RUN SAID")
print("run id      : %s" % get("_meta.run_id", RUN_ID))
print("run mode    : %s" % get("env.run_mode", "?"))
print("digest      : %s" % ("complete" if get("_meta.complete") else
                            "INCOMPLETE, missing %s" % get("_meta.missing_sections")))

s_over = get("sanity.overfit") or {}
s_res  = get("sanity.resume") or {}
s_spd  = get("sanity.speed") or {}
if s_over or s_res or s_spd:
    print("")
    print("SANITY VERDICTS")
    if s_over:
        print("  overfit 32      : %s   loss %.4f, WER on its own targets %.4f"
              % ("PASS" if s_over.get("passed") else "FAIL",
                 s_over.get("final_loss", float("nan")), s_over.get("wer_on_own_targets", float("nan"))))
    if s_res:
        print("  resume in-proc  : %s   (optimizer %s, scheduler %s, step %s)"
              % ("PASS" if s_res.get("in_process_passed") else "FAIL",
                 s_res.get("optimizer_state"), s_res.get("scheduler_state"), s_res.get("steps_after")))
        print("  resume x-session: %s" % ("PASS" if s_res.get("cross_session_passed") else "NOT YET PROVEN"))
    if s_spd:
        print("  speed           : %.3f s per micro-batch, %.3f s per generated clip, peak VRAM %.1f GB"
              % (s_spd.get("sec_per_microbatch", float("nan")),
                 s_spd.get("sec_per_gen_clip", float("nan")), s_spd.get("peak_vram_gb", float("nan"))))
        print("  projection      : %s steps, %.2f h total  -> %s"
              % (s_spd.get("n_steps"), s_spd.get("total_hours", float("nan")),
                 "FITS the budget" if s_spd.get("fits_budget") else "EXCEEDS the budget"))

v = get("validation") or {}
if v:
    print("")
    print("SELF-VALIDATION: %s/%s" % (v.get("n_passed"), v.get("n_total")))
    for c in (v.get("checks") or []):
        if not c.get("passed"):
            print("  FAIL  %-58s %s" % (c.get("check"), c.get("detail", ""))[:150])

head("DIGEST")
if D:
    blob = json.dumps(D, ensure_ascii=False, separators=(",", ":"), sort_keys=True)
    print("===== BEGIN CLAUDE-HANDOFF-JSON =====")
    print(blob)
    print("===== END CLAUDE-HANDOFF-JSON =====")
    print("")
    print("size: %.1f KB - select the whole block above and paste it into the chat" % (len(blob) / 1024))
else:
    print("dataset_findings.json was not in the extracted folder. Files present:")
    for p in sorted(DEST.rglob("*")):
        if p.is_file():
            print("  %s" % p.relative_to(DEST))